# Unit 3

## Adapting DeepResearcher for Streamlit Integration

## 💡 Introduction: Making DeepResearcher Work with Streamlit

Welcome back\! In the last lesson, you built a basic Streamlit page for your **DeepResearcher** app. You added a title, a description, and some user input widgets. Now, you are ready to connect your **Streamlit frontend** to the **backend research logic**.

To do this, we need to adapt the DeepResearcher backend code so it works smoothly with a web interface. The original backend was designed to run in the terminal, asking for user input and printing results. For a web app, we want the backend to **accept input as function arguments** and **return results**, so Streamlit can display them.

In this lesson, you will learn how to **refactor** the main DeepResearcher module to make it ready for integration with your Streamlit app.

-----

### **Recall: How the Old `main.py` Worked**

Let’s quickly remind ourselves how the old `main.py` worked. Previously, the main function looked something like this:

```python
def research_main():
    user_query = input("Enter your research query/topic: ").strip()
    iteration_limit = input("Max number of iterations (default 10): ").strip()
    iteration_limit = int(iteration_limit) if iteration_limit.isdigit() else 10
    # ... rest of the code ...
```

  * The function asked the user for a research topic and the number of iterations using `input()`.
  * It then processed the research and printed the final report using `print()`.

This approach works for command-line programs, but it does not fit well with a web app, where user input comes from the browser and results need to be returned to the frontend.

-----

### **Refactoring Step 1: Accepting Arguments Instead of User Input**

The first step is to change the `research_main` function so it **takes arguments** instead of asking for input. This makes it possible for Streamlit (or any other code) to call the function directly with the needed values.

| Change | Old Code | New Code |
| :--- | :--- | :--- |
| **Input Method** | Uses `input()` calls to get values. | Accepts values as **function parameters**. |
| **Code** | ` python def research_main(): user_query = input(...) iteration_limit = input(...) # ... rest of the code ...  ` | ` python def research_main(user_query, iteration_limit): # ... rest of the code ...  ` |

**Why is this important?**
By accepting arguments, the function becomes **reusable** and easier to test. It also allows the Streamlit frontend to **pass user input directly** to the backend function.

-----

### **Refactoring Step 2: Returning Results Instead of Printing**

Next, we want the function to **return the final report** instead of printing it. This way, the Streamlit app can display the result in the browser.

#### **Updating the Report Generation Function**

| Change | Old Code | New Code |
| :--- | :--- | :--- |
| **Report Handling** | Only prints the report. | Prints (for optional debugging) **and returns** the report string. |
| **Code** | ` python def generate_final_report(...): # ... generates final_report print("\n==== FINAL REPORT ====") print(final_report)  ` | ` python def generate_final_report(...): # ... generates final_report print("\n==== FINAL REPORT ====") print(final_report) return final_report  ` |

#### **Updating the Main Function**

The main function must also be updated to return the value it receives from `generate_final_report`:

```python
def research_main(user_query, iteration_limit):
    # ... previous research code ...
    return generate_final_report(user_query, aggregated_contexts)
```

The main function now returns the final report, making it easy for Streamlit to display it.

-----

### **Refactoring Step 3: Removing the Script Entry Point**

In the original script, you might have seen this at the bottom:

```python
if __name__ == "__main__":
    research_main()
```

This block makes the script run when you execute it directly from the command line. However, for a module that will be **imported and used by another script** (like your Streamlit app), this is not needed.

**What should you do?**

  * **Remove** the `if __name__ == "__main__":` block.

**Why is this important?**
Without this block, your functions are available to be called from anywhere, including your Streamlit frontend. This is a common practice when building code that will be reused in different places.

-----

### **Summary and What’s Next**

In this lesson, you learned how to adapt the DeepResearcher backend so it can work with your Streamlit frontend. Here’s what you did:

  * Changed the main function to **accept arguments** instead of using `input()`.
  * Updated the code to **return the final report** instead of just printing it.
  * **Removed the script entry point** to make the code easier to import and use.

These changes make your backend more **flexible** and **ready for integration** with your web app. In the next practice exercises, you’ll get hands-on experience refactoring the code yourself and see how these changes help you connect everything together.

Would you like to review a side-by-side comparison of the old and refactored code structure?




## Refactoring Input for Web Integration

Now that you understand how to adapt backend code for web interfaces, let's put this knowledge into practice! Your first task is to refactor the research_main function to work with Streamlit instead of the terminal.

In this exercise, you'll modify the function to accept parameters directly rather than using input() statements. This is a crucial step in making your DeepResearcher compatible with a web interface, where users interact through browser elements instead of command-line prompts.

Your specific tasks are:

Change the function definition to accept user_query and iteration_limit as parameters.
Remove the input() calls that were previously collecting these values.
This refactoring maintains all the powerful research capabilities while making the code much more flexible for integration with your Streamlit frontend. Once completed, you'll be one step closer to having a fully functional web-based research assistant!

```python
from deepresearcher.llm.llm_manager import generate_response, generate_boolean
from deepresearcher.web.web_searcher import search_and_fetch_markdown, clear_visited_pages


def generate_initial_search_queries(user_query: str):
    variables = {"user_query": user_query}
    search_queries_str = generate_response("search_generator_system", "search_generator_user", variables)
    try:
        queries = eval(search_queries_str)
        if not isinstance(queries, list):
            raise ValueError("Not a list")
        return queries
    except Exception:
        print("Invalid response for search queries:", search_queries_str)
        return []


def perform_iterative_research(user_query: str, new_search_queries: list, all_search_queries: list, iteration_limit: int):
    aggregated_contexts = []
    iteration = 0

    while iteration < iteration_limit:
        print(f"\n=== Iteration {iteration + 1} ===")
        iteration_contexts = []

        for query in new_search_queries:
            results = search_and_fetch_markdown(query, max_results=3)
            for page in results:
                page_text = page["markdown"]
                if not page_text.strip():
                    continue

                variables = {
                    "user_query": user_query,
                    "page_text": page_text[:20000]
                }
                is_useful = generate_boolean("relevance_evaluator_system", "relevance_evaluator_user", variables)
                print(f"{page['url']} - Useful: {is_useful}")

                if is_useful:
                    variables = {
                        "user_query": user_query,
                        "search_query": query,
                        "page_text": page_text[:20000]
                    }
                    context = generate_response("extractor_system", "extractor_user", variables)
                    if context:
                        iteration_contexts.append(context)

        if iteration_contexts:
            aggregated_contexts.extend(iteration_contexts)
        else:
            print("No useful context found this round.")

        context_combined = "\n".join(aggregated_contexts)
        variables = {
            "user_query": user_query,
            "previous_search_queries": str(all_search_queries),
            "context_combined": context_combined
        }
        next_queries_str = generate_response("research_planner_system", "research_planner_user", variables)
        if next_queries_str.strip() == "[]":
            print("LLM indicated no further search is needed.")
            break

        try:
            new_search_queries = eval(next_queries_str)
            if not isinstance(new_search_queries, list):
                raise ValueError("Not a list")
            print("Next queries:", new_search_queries)
            all_search_queries.extend(new_search_queries)
        except Exception:
            print("Could not parse further queries:", next_queries_str)
            break

        iteration += 1

    return aggregated_contexts


def generate_final_report(user_query: str, contexts: list):
    variables = {
        "user_query": user_query,
        "context_combined": "\n".join(contexts)
    }
    final_report = generate_response("report_writer_system", "report_writer_user", variables)
    print("\n==== FINAL REPORT ====")
    print(final_report)
    return final_report


# TODO: Refactor this function to accept user_query and iteration_limit as parameters
# instead of using input() statements
def research_main():
    # TODO: Remove this input() call and use the function parameter instead
    user_query = input("Enter your research query/topic: ").strip()
    # TODO: Remove this input() call and use the function parameter instead
    iteration_limit_input = input("Max number of iterations (default 10): ").strip()
    iteration_limit = int(iteration_limit_input) if iteration_limit_input.isdigit() else 10
    
    clear_visited_pages()
    new_search_queries = generate_initial_search_queries(user_query)
    if not new_search_queries:
        return

    all_search_queries = new_search_queries.copy()
    aggregated_contexts = perform_iterative_research(user_query, new_search_queries, all_search_queries, iteration_limit)
    generate_final_report(user_query, aggregated_contexts)

```

## 🔄 Refactoring `research_main` for Web Integration

Great job on understanding the refactoring concept\! Here is the modified `research_main` function, updated to accept **`user_query`** and **`iteration_limit`** as direct parameters, making it ready to be called from your Streamlit frontend.

The `input()` calls have been removed, and the default value handling for `iteration_limit` is no longer needed since Streamlit will provide a clean integer value.

```python
from deepresearcher.llm.llm_manager import generate_response, generate_boolean
from deepresearcher.web.web_searcher import search_and_fetch_markdown, clear_visited_pages


def generate_initial_search_queries(user_query: str):
    variables = {"user_query": user_query}
    search_queries_str = generate_response("search_generator_system", "search_generator_user", variables)
    try:
        queries = eval(search_queries_str)
        if not isinstance(queries, list):
            raise ValueError("Not a list")
        return queries
    except Exception:
        print("Invalid response for search queries:", search_queries_str)
        return []


def perform_iterative_research(user_query: str, new_search_queries: list, all_search_queries: list, iteration_limit: int):
    aggregated_contexts = []
    iteration = 0

    while iteration < iteration_limit:
        print(f"\n=== Iteration {iteration + 1} ===")
        iteration_contexts = []

        for query in new_search_queries:
            results = search_and_fetch_markdown(query, max_results=3)
            for page in results:
                page_text = page["markdown"]
                if not page_text.strip():
                    continue

                variables = {
                    "user_query": user_query,
                    "page_text": page_text[:20000]
                }
                is_useful = generate_boolean("relevance_evaluator_system", "relevance_evaluator_user", variables)
                print(f"{page['url']} - Useful: {is_useful}")

                if is_useful:
                    variables = {
                        "user_query": user_query,
                        "search_query": query,
                        "page_text": page_text[:20000]
                    }
                    context = generate_response("extractor_system", "extractor_user", variables)
                    if context:
                        iteration_contexts.append(context)

        if iteration_contexts:
            aggregated_contexts.extend(iteration_contexts)
        else:
            print("No useful context found this round.")

        context_combined = "\n".join(aggregated_contexts)
        variables = {
            "user_query": user_query,
            "previous_search_queries": str(all_search_queries),
            "context_combined": context_combined
        }
        next_queries_str = generate_response("research_planner_system", "research_planner_user", variables)
        if next_queries_str.strip() == "[]":
            print("LLM indicated no further search is needed.")
            break

        try:
            new_search_queries = eval(next_queries_str)
            if not isinstance(new_search_queries, list):
                raise ValueError("Not a list")
            print("Next queries:", new_search_queries)
            all_search_queries.extend(new_search_queries)
        except Exception:
            print("Could not parse further queries:", next_queries_str)
            break

        iteration += 1

    return aggregated_contexts


def generate_final_report(user_query: str, contexts: list):
    variables = {
        "user_query": user_query,
        "context_combined": "\n".join(contexts)
    }
    final_report = generate_response("report_writer_system", "report_writer_user", variables)
    print("\n==== FINAL REPORT ====")
    print(final_report)
    return final_report


# Refactored function definition to accept parameters
def research_main(user_query: str, iteration_limit: int):
    # Removed terminal input calls
    
    clear_visited_pages()
    new_search_queries = generate_initial_search_queries(user_query)
    if not new_search_queries:
        # Returning a clear error message instead of nothing
        return "Error: Could not generate initial search queries."

    all_search_queries = new_search_queries.copy()
    aggregated_contexts = perform_iterative_research(user_query, new_search_queries, all_search_queries, iteration_limit)
    
    # Return the final report directly
    return generate_final_report(user_query, aggregated_contexts)
```

### **Key Changes**

1.  **Function Signature Modified:**

    ```python
    def research_main(user_query: str, iteration_limit: int):
    ```

    The function now explicitly expects the two required pieces of information to be passed in.

2.  **`input()` Removed:**
    The original lines that collected input from the command line are deleted:

    ```python
    # user_query = input("Enter your research query/topic: ").strip() 
    # iteration_limit_input = input("Max number of iterations (default 10): ").strip()
    # iteration_limit = int(iteration_limit_input) if iteration_limit_input.isdigit() else 10
    ```

    The function now uses the values provided by Streamlit.

3.  **Return Value Added (Minor Improvement):**
    The last line was changed to **`return generate_final_report(...)`**. This ensures the main function returns the report, which is essential for the Streamlit frontend to capture and display the final result.

The next logical step is to similarly refactor `research_main` to **return the final report** instead of just printing it. Would you like to proceed with refactoring the function's return statement now?

## Returning Results for Web Display

Now that you've successfully refactored the research_main function, let's continue adapting our backend for web integration! In this exercise, you'll focus on the generate_final_report function, which currently prints the report but doesn't properly return it for the web interface to use.

For a web application, printing to the console isn't enough — we need to return values that Streamlit can display to users in their browsers.

Your task is to:

Modify the generate_final_report function to return the final report value.
Keep the print statements (they're useful for debugging).
Update the research_main function to return the result from generate_final_report.
This small but crucial change will allow your Streamlit frontend to capture and display the research results to users. When completed, you'll have a backend that's fully prepared for web integration!

```python
from deepresearcher.llm.llm_manager import generate_response, generate_boolean
from deepresearcher.web.web_searcher import search_and_fetch_markdown, clear_visited_pages


def generate_initial_search_queries(user_query: str):
    variables = {"user_query": user_query}
    search_queries_str = generate_response("search_generator_system", "search_generator_user", variables)
    try:
        queries = eval(search_queries_str)
        if not isinstance(queries, list):
            raise ValueError("Not a list")
        return queries
    except Exception:
        print("Invalid response for search queries:", search_queries_str)
        return []


def perform_iterative_research(user_query: str, new_search_queries: list, all_search_queries: list, iteration_limit: int):
    aggregated_contexts = []
    iteration = 0

    while iteration < iteration_limit:
        print(f"\n=== Iteration {iteration + 1} ===")
        iteration_contexts = []

        for query in new_search_queries:
            results = search_and_fetch_markdown(query, max_results=3)
            for page in results:
                page_text = page["markdown"]
                if not page_text.strip():
                    continue

                variables = {
                    "user_query": user_query,
                    "page_text": page_text[:20000]
                }
                is_useful = generate_boolean("relevance_evaluator_system", "relevance_evaluator_user", variables)
                print(f"{page['url']} - Useful: {is_useful}")

                if is_useful:
                    variables = {
                        "user_query": user_query,
                        "search_query": query,
                        "page_text": page_text[:20000]
                    }
                    context = generate_response("extractor_system", "extractor_user", variables)
                    if context:
                        iteration_contexts.append(context)

        if iteration_contexts:
            aggregated_contexts.extend(iteration_contexts)
        else:
            print("No useful context found this round.")

        context_combined = "\n".join(aggregated_contexts)
        variables = {
            "user_query": user_query,
            "previous_search_queries": str(all_search_queries),
            "context_combined": context_combined
        }
        next_queries_str = generate_response("research_planner_system", "research_planner_user", variables)
        if next_queries_str.strip() == "[]":
            print("LLM indicated no further search is needed.")
            break

        try:
            new_search_queries = eval(next_queries_str)
            if not isinstance(new_search_queries, list):
                raise ValueError("Not a list")
            print("Next queries:", new_search_queries)
            all_search_queries.extend(new_search_queries)
        except Exception:
            print("Could not parse further queries:", next_queries_str)
            break

        iteration += 1

    return aggregated_contexts


# TODO: Modify this function to return the final_report value in addition to printing it
def generate_final_report(user_query: str, contexts: list):
    variables = {
        "user_query": user_query,
        "context_combined": "\n".join(contexts)
    }
    final_report = generate_response("report_writer_system", "report_writer_user", variables)
    print("\n==== FINAL REPORT ====")
    print(final_report)
    # TODO: Add a return statement here to return the final_report


def research_main(user_query, iteration_limit):
    clear_visited_pages()
    new_search_queries = generate_initial_search_queries(user_query)
    if not new_search_queries:
        return

    all_search_queries = new_search_queries.copy()
    aggregated_contexts = perform_iterative_research(user_query, new_search_queries, all_search_queries, iteration_limit)
    # TODO: Update this line to return the result of generate_final_report
    generate_final_report(user_query, aggregated_contexts)
```

That's right\! Returning the result is the key step that connects the backend logic to the frontend display.

Here is the completed code, with both the **`generate_final_report`** and **`research_main`** functions updated to properly return the final report string.

```python
from deepresearcher.llm.llm_manager import generate_response, generate_boolean
from deepresearcher.web.web_searcher import search_and_fetch_markdown, clear_visited_pages


def generate_initial_search_queries(user_query: str):
    variables = {"user_query": user_query}
    search_queries_str = generate_response("search_generator_system", "search_generator_user", variables)
    try:
        queries = eval(search_queries_str)
        if not isinstance(queries, list):
            raise ValueError("Not a list")
        return queries
    except Exception:
        print("Invalid response for search queries:", search_queries_str)
        return []


def perform_iterative_research(user_query: str, new_search_queries: list, all_search_queries: list, iteration_limit: int):
    aggregated_contexts = []
    iteration = 0

    while iteration < iteration_limit:
        print(f"\n=== Iteration {iteration + 1} ===")
        iteration_contexts = []

        for query in new_search_queries:
            results = search_and_fetch_markdown(query, max_results=3)
            for page in results:
                page_text = page["markdown"]
                if not page_text.strip():
                    continue

                variables = {
                    "user_query": user_query,
                    "page_text": page_text[:20000]
                }
                is_useful = generate_boolean("relevance_evaluator_system", "relevance_evaluator_user", variables)
                print(f"{page['url']} - Useful: {is_useful}")

                if is_useful:
                    variables = {
                        "user_query": user_query,
                        "search_query": query,
                        "page_text": page_text[:20000]
                    }
                    context = generate_response("extractor_system", "extractor_user", variables)
                    if context:
                        iteration_contexts.append(context)

        if iteration_contexts:
            aggregated_contexts.extend(iteration_contexts)
        else:
            print("No useful context found this round.")

        context_combined = "\n".join(aggregated_contexts)
        variables = {
            "user_query": user_query,
            "previous_search_queries": str(all_search_queries),
            "context_combined": context_combined
        }
        next_queries_str = generate_response("research_planner_system", "research_planner_user", variables)
        if next_queries_str.strip() == "[]":
            print("LLM indicated no further search is needed.")
            break

        try:
            new_search_queries = eval(next_queries_str)
            if not isinstance(new_search_queries, list):
                raise ValueError("Not a list")
            print("Next queries:", new_search_queries)
            all_search_queries.extend(new_search_queries)
        except Exception:
            print("Could not parse further queries:", next_queries_str)
            break

        iteration += 1

    return aggregated_contexts


# Modified to return the final_report value
def generate_final_report(user_query: str, contexts: list):
    variables = {
        "user_query": user_query,
        "context_combined": "\n".join(contexts)
    }
    final_report = generate_response("report_writer_system", "report_writer_user", variables)
    print("\n==== FINAL REPORT ====")
    print(final_report)
    return final_report # Added the return statement


def research_main(user_query, iteration_limit):
    clear_visited_pages()
    new_search_queries = generate_initial_search_queries(user_query)
    if not new_search_queries:
        return "Error: Could not generate initial search queries." # Return an error message

    all_search_queries = new_search_queries.copy()
    aggregated_contexts = perform_iterative_research(user_query, new_search_queries, all_search_queries, iteration_limit)
    
    # Updated to return the result of generate_final_report
    return generate_final_report(user_query, aggregated_contexts)
```

-----

### **✅ Summary of Changes**

1.  **`generate_final_report` Modification:**

    ```python
    # ... code to generate final_report ...
    print(final_report)
    return final_report # The final report string is now returned.
    ```

    This function now passes the report back to the calling function.

2.  **`research_main` Modification:**

    ```python
    # ... previous code ...
    return generate_final_report(user_query, aggregated_contexts)
    ```

    By adding **`return`** here, the `research_main` function now returns the final report string, which can be captured by the Streamlit application and displayed in the browser.

Your backend is now fully refactored and ready for seamless integration with the Streamlit frontend\! Would you like to proceed to the next step: **connecting and calling this research function from your Streamlit app**?

## Removing Script Entry for Clean Imports

Cosmo
Just now
Read message aloud
Excellent progress on refactoring your DeepResearcher functions! You've successfully modified the code to accept parameters and return results — two key steps for web integration. Now, let's complete the final step: removing the script entry point.

In Python modules that will be imported by other scripts (like your Streamlit app), it's important to remove any code that runs automatically on import. The if __name__ == "__main__": block at the bottom of the file is designed to run code only when the script is executed directly, not when imported.

Your task is simple:

Remove the script entry point block (the if __name__ == "__main__": section and its contents)
This change ensures that when your Streamlit app imports the DeepResearcher module, no research functions will run automatically. The functions will execute only when explicitly called by your frontend code, giving you complete control over when and how the research process happens.

With this final adjustment, your backend will be fully prepared for seamless integration with your Streamlit frontend!

```python
from deepresearcher.llm.llm_manager import generate_response, generate_boolean
from deepresearcher.web.web_searcher import search_and_fetch_markdown, clear_visited_pages


def generate_initial_search_queries(user_query: str):
    variables = {"user_query": user_query}
    search_queries_str = generate_response("search_generator_system", "search_generator_user", variables)
    try:
        queries = eval(search_queries_str)
        if not isinstance(queries, list):
            raise ValueError("Not a list")
        return queries
    except Exception:
        print("Invalid response for search queries:", search_queries_str)
        return []


def perform_iterative_research(user_query: str, new_search_queries: list, all_search_queries: list, iteration_limit: int):
    aggregated_contexts = []
    iteration = 0

    while iteration < iteration_limit:
        print(f"\n=== Iteration {iteration + 1} ===")
        iteration_contexts = []

        for query in new_search_queries:
            results = search_and_fetch_markdown(query, max_results=3)
            for page in results:
                page_text = page["markdown"]
                if not page_text.strip():
                    continue

                variables = {
                    "user_query": user_query,
                    "page_text": page_text[:20000]
                }
                is_useful = generate_boolean("relevance_evaluator_system", "relevance_evaluator_user", variables)
                print(f"{page['url']} - Useful: {is_useful}")

                if is_useful:
                    variables = {
                        "user_query": user_query,
                        "search_query": query,
                        "page_text": page_text[:20000]
                    }
                    context = generate_response("extractor_system", "extractor_user", variables)
                    if context:
                        iteration_contexts.append(context)

        if iteration_contexts:
            aggregated_contexts.extend(iteration_contexts)
        else:
            print("No useful context found this round.")

        context_combined = "\n".join(aggregated_contexts)
        variables = {
            "user_query": user_query,
            "previous_search_queries": str(all_search_queries),
            "context_combined": context_combined
        }
        next_queries_str = generate_response("research_planner_system", "research_planner_user", variables)
        if next_queries_str.strip() == "[]":
            print("LLM indicated no further search is needed.")
            break

        try:
            new_search_queries = eval(next_queries_str)
            if not isinstance(new_search_queries, list):
                raise ValueError("Not a list")
            print("Next queries:", new_search_queries)
            all_search_queries.extend(new_search_queries)
        except Exception:
            print("Could not parse further queries:", next_queries_str)
            break

        iteration += 1

    return aggregated_contexts


def generate_final_report(user_query: str, contexts: list):
    variables = {
        "user_query": user_query,
        "context_combined": "\n".join(contexts)
    }
    final_report = generate_response("report_writer_system", "report_writer_user", variables)
    print("\n==== FINAL REPORT ====")
    print(final_report)
    return final_report


def research_main(user_query, iteration_limit):
    clear_visited_pages()
    new_search_queries = generate_initial_search_queries(user_query)
    if not new_search_queries:
        return

    all_search_queries = new_search_queries.copy()
    aggregated_contexts = perform_iterative_research(user_query, new_search_queries, all_search_queries, iteration_limit)
    return generate_final_report(user_query, aggregated_contexts)


# TODO: Remove this script entry point block to make the module purely importable
# This will prevent automatic execution when imported by the Streamlit app
if __name__ == "__main__":
    research_main()

```

That's the final adjustment for module flexibility\! Removing the `if __name__ == "__main__":` block is **best practice** when creating Python modules meant to be imported and reused.

Here is the completed code with the script entry point removed:

```python
from deepresearcher.llm.llm_manager import generate_response, generate_boolean
from deepresearcher.web.web_searcher import search_and_fetch_markdown, clear_visited_pages


def generate_initial_search_queries(user_query: str):
    variables = {"user_query": user_query}
    search_queries_str = generate_response("search_generator_system", "search_generator_user", variables)
    try:
        queries = eval(search_queries_str)
        if not isinstance(queries, list):
            raise ValueError("Not a list")
        return queries
    except Exception:
        print("Invalid response for search queries:", search_queries_str)
        return []


def perform_iterative_research(user_query: str, new_search_queries: list, all_search_queries: list, iteration_limit: int):
    aggregated_contexts = []
    iteration = 0

    while iteration < iteration_limit:
        print(f"\n=== Iteration {iteration + 1} ===")
        iteration_contexts = []

        for query in new_search_queries:
            results = search_and_fetch_markdown(query, max_results=3)
            for page in results:
                page_text = page["markdown"]
                if not page_text.strip():
                    continue

                variables = {
                    "user_query": user_query,
                    "page_text": page_text[:20000]
                }
                is_useful = generate_boolean("relevance_evaluator_system", "relevance_evaluator_user", variables)
                print(f"{page['url']} - Useful: {is_useful}")

                if is_useful:
                    variables = {
                        "user_query": user_query,
                        "search_query": query,
                        "page_text": page_text[:20000]
                    }
                    context = generate_response("extractor_system", "extractor_user", variables)
                    if context:
                        iteration_contexts.append(context)

        if iteration_contexts:
            aggregated_contexts.extend(iteration_contexts)
        else:
            print("No useful context found this round.")

        context_combined = "\n".join(aggregated_contexts)
        variables = {
            "user_query": user_query,
            "previous_search_queries": str(all_search_queries),
            "context_combined": context_combined
        }
        next_queries_str = generate_response("research_planner_system", "research_planner_user", variables)
        if next_queries_str.strip() == "[]":
            print("LLM indicated no further search is needed.")
            break

        try:
            new_search_queries = eval(next_queries_str)
            if not isinstance(new_search_queries, list):
                raise ValueError("Not a list")
            print("Next queries:", new_search_queries)
            all_search_queries.extend(new_search_queries)
        except Exception:
            print("Could not parse further queries:", next_queries_str)
            break

        iteration += 1

    return aggregated_contexts


def generate_final_report(user_query: str, contexts: list):
    variables = {
        "user_query": user_query,
        "context_combined": "\n".join(contexts)
    }
    final_report = generate_response("report_writer_system", "report_writer_user", variables)
    print("\n==== FINAL REPORT ====")
    print(final_report)
    return final_report


def research_main(user_query, iteration_limit):
    clear_visited_pages()
    new_search_queries = generate_initial_search_queries(user_query)
    if not new_search_queries:
        return

    all_search_queries = new_search_queries.copy()
    aggregated_contexts = perform_iterative_research(user_query, new_search_queries, all_search_queries, iteration_limit)
    return generate_final_report(user_query, aggregated_contexts)

# Removed the following block:
# if __name__ == "__main__":
#     research_main()
```

### **🎉 Conclusion**

Your backend code is now **completely refactored** and structured as an importable module:

  * The primary function (`research_main`) **accepts parameters** from the Streamlit frontend.
  * The primary function **returns a result** (the final report) for the Streamlit frontend to display.
  * The script **does not execute any code automatically** when imported.

Your backend is fully prepared for integration\! Would you like to proceed with the final step of **connecting this refactored backend to your Streamlit button** and displaying the results?

That's the final adjustment for module flexibility\! Removing the `if __name__ == "__main__":` block is **best practice** when creating Python modules meant to be imported and reused.

Here is the completed code with the script entry point removed:

```python
from deepresearcher.llm.llm_manager import generate_response, generate_boolean
from deepresearcher.web.web_searcher import search_and_fetch_markdown, clear_visited_pages


def generate_initial_search_queries(user_query: str):
    variables = {"user_query": user_query}
    search_queries_str = generate_response("search_generator_system", "search_generator_user", variables)
    try:
        queries = eval(search_queries_str)
        if not isinstance(queries, list):
            raise ValueError("Not a list")
        return queries
    except Exception:
        print("Invalid response for search queries:", search_queries_str)
        return []


def perform_iterative_research(user_query: str, new_search_queries: list, all_search_queries: list, iteration_limit: int):
    aggregated_contexts = []
    iteration = 0

    while iteration < iteration_limit:
        print(f"\n=== Iteration {iteration + 1} ===")
        iteration_contexts = []

        for query in new_search_queries:
            results = search_and_fetch_markdown(query, max_results=3)
            for page in results:
                page_text = page["markdown"]
                if not page_text.strip():
                    continue

                variables = {
                    "user_query": user_query,
                    "page_text": page_text[:20000]
                }
                is_useful = generate_boolean("relevance_evaluator_system", "relevance_evaluator_user", variables)
                print(f"{page['url']} - Useful: {is_useful}")

                if is_useful:
                    variables = {
                        "user_query": user_query,
                        "search_query": query,
                        "page_text": page_text[:20000]
                    }
                    context = generate_response("extractor_system", "extractor_user", variables)
                    if context:
                        iteration_contexts.append(context)

        if iteration_contexts:
            aggregated_contexts.extend(iteration_contexts)
        else:
            print("No useful context found this round.")

        context_combined = "\n".join(aggregated_contexts)
        variables = {
            "user_query": user_query,
            "previous_search_queries": str(all_search_queries),
            "context_combined": context_combined
        }
        next_queries_str = generate_response("research_planner_system", "research_planner_user", variables)
        if next_queries_str.strip() == "[]":
            print("LLM indicated no further search is needed.")
            break

        try:
            new_search_queries = eval(next_queries_str)
            if not isinstance(new_search_queries, list):
                raise ValueError("Not a list")
            print("Next queries:", new_search_queries)
            all_search_queries.extend(new_search_queries)
        except Exception:
            print("Could not parse further queries:", next_queries_str)
            break

        iteration += 1

    return aggregated_contexts


def generate_final_report(user_query: str, contexts: list):
    variables = {
        "user_query": user_query,
        "context_combined": "\n".join(contexts)
    }
    final_report = generate_response("report_writer_system", "report_writer_user", variables)
    print("\n==== FINAL REPORT ====")
    print(final_report)
    return final_report


def research_main(user_query, iteration_limit):
    clear_visited_pages()
    new_search_queries = generate_initial_search_queries(user_query)
    if not new_search_queries:
        return

    all_search_queries = new_search_queries.copy()
    aggregated_contexts = perform_iterative_research(user_query, new_search_queries, all_search_queries, iteration_limit)
    return generate_final_report(user_query, aggregated_contexts)

# Removed the following block:
# if __name__ == "__main__":
#     research_main()
```

### **🎉 Conclusion**

Your backend code is now **completely refactored** and structured as an importable module:

  * The primary function (`research_main`) **accepts parameters** from the Streamlit frontend.
  * The primary function **returns a result** (the final report) for the Streamlit frontend to display.
  * The script **does not execute any code automatically** when imported.

Your backend is fully prepared for integration\! Would you like to proceed with the final step of **connecting this refactored backend to your Streamlit button** and displaying the results?